# 04 - Entrenamiento YOLOv26 con MLflow

Entrena un modelo YOLOv26 nano para deteccion de juguetes (Captain / Gohan)
usando datos validados del pipeline medallion.

**Flujo:**
1. Consultar Feature Store para validar calidad del dataset
2. Entrenar modelo YOLO con GPU
3. Registrar parametros, metricas y artefactos en MLflow
4. Registrar modelo en Unity Catalog

**Prerequisitos:** Ejecutar notebooks 01, 02 y 03

In [0]:
%pip install numpy==1.26.4 ultralytics mlflow databricks-feature-engineering -q
%pip uninstall -y opencv-python -q
%pip install --force-reinstall --no-deps opencv-python-headless==4.10.0.84 -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
databricks-serverless-gpu 0.5.11+patch3 requires databricks-connect<16,>=15.4.2, but you have databricks-connect 17.3.7 which is incompatible.
databricks-serverless-gpu 0.5.11+patch3 requires mlflow<3.0,>=2.17, but you have mlflow 3.11.1 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import os
import yaml
import mlflow
import pandas as pd
from pyspark.sql import functions as F

# Configurar directorio de ultralytics antes de importar (evita PermissionError en serverless)
os.environ["YOLO_CONFIG_DIR"] = "/tmp/ultralytics_cfg"
os.makedirs("/tmp/ultralytics_cfg", exist_ok=True)

from ultralytics import YOLO

# -- Unity Catalog --
CATALOG = "jgworkspaceclassic_catalog"
SCHEMA = "infra_demo"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.cv_v2_silver"
GOLD_IMAGE_TABLE = f"{CATALOG}.{SCHEMA}.cv_v2_gold_image_features"
GOLD_ANNOTATION_TABLE = f"{CATALOG}.{SCHEMA}.cv_v2_gold_annotation_features"

# -- Rutas --
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/computer_vision"

# -- Modelo --
YOLO_MODEL = "yolo26n.pt"

# -- Hiperparametros --
EPOCHS = 50
PATIENCE = 10
IMG_SIZE = 640
BATCH_SIZE = 16

# -- MLflow --
EXPERIMENT_NAME = "/Users/juan.gordon@databricks.com/GrupoInfraCV/v2/cv-toys-detection"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.cv_v2_yolo_toys"

/databricks/python/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/tmp/ultralytics_cfg/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

# Leer features desde Feature Store
df_image_features = spark.table(GOLD_IMAGE_TABLE)
df_annotation_features = spark.table(GOLD_ANNOTATION_TABLE)

print("=" * 55)
print("VALIDACION DEL DATASET (desde Feature Store)")
print("=" * 55)

# Imagenes por split
print("\nImagenes por split:")
display(
    df_image_features
    .groupBy("split")
    .agg(
        F.count("*").alias("images"),
        F.sum("num_objects").alias("total_objects"),
        F.avg("num_objects").cast("decimal(4,2)").alias("avg_objects_per_image"),
        F.avg("avg_bbox_area").cast("decimal(6,4)").alias("avg_bbox_area"),
    )
    .orderBy("split")
)

# Distribucion de tamanos
print("\nDistribucion de tamanos de objetos:")
display(
    df_annotation_features
    .groupBy("split", "size_category")
    .agg(F.count("*").alias("count"))
    .orderBy("split", "size_category")
)

# Verificar calidad
df_silver = spark.table(SILVER_TABLE)
total = df_silver.count()
valid = df_silver.filter(F.col("is_valid")).count()
quality_ratio = valid / total if total > 0 else 0

print(f"\nCalidad: {valid}/{total} ({quality_ratio:.1%})")
assert quality_ratio >= 0.8, f"Dataset con calidad insuficiente: {quality_ratio:.1%}"
print("Dataset aprobado para entrenamiento")

VALIDACION DEL DATASET (desde Feature Store)

Imagenes por split:


split,images,total_objects,avg_objects_per_image,avg_bbox_area
test,7,8,1.14,0.3273
train,153,165,1.08,0.3038
valid,15,16,1.07,0.3027



Distribucion de tamanos de objetos:


split,size_category,count
test,very_large,8
train,large,20
train,very_large,145
valid,large,1
valid,very_large,15



Calidad: 189/189 (100.0%)
Dataset aprobado para entrenamiento


In [0]:
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if exp is None:
    mlflow.create_experiment(name=EXPERIMENT_NAME)
    print(f"Experimento CREADO: {EXPERIMENT_NAME}")
elif exp.lifecycle_stage == "deleted":
    client = mlflow.tracking.MlflowClient()
    client.restore_experiment(exp.experiment_id)
    print(f"Experimento RESTAURADO: {EXPERIMENT_NAME}")
else:
    print(f"Experimento existente: {EXPERIMENT_NAME}")

mlflow.set_experiment(EXPERIMENT_NAME)

Experimento existente: /Users/juan.gordon@databricks.com/GrupoInfraCV/v2/cv-toys-detection


<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/2564958131611974', creation_time=1776906265103, experiment_id='2564958131611974', last_update_time=1776906265103, lifecycle_stage='active', name='/Users/juan.gordon@databricks.com/GrupoInfraCV/v2/cv-toys-detection', tags={'mlflow.experiment.sourceName': '/Users/juan.gordon@databricks.com/GrupoInfraCV/v2/cv-toys-detection',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'andrea.estaire@databricks.com',
 'mlflow.ownerId': '2117503826472987'}, trace_location=None, workspace='default'>

In [0]:
DATA_YAML = os.path.join(VOLUME_PATH, "data.yaml")

# Desactivar callback MLflow nativo de Ultralytics
_original_tracking_uri = os.environ.pop("MLFLOW_TRACKING_URI", None)

# Entrenar
model = YOLO(YOLO_MODEL)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project="/tmp/yolo_training_v2",
    name="toys-detection-v2",
    exist_ok=True,
)

# Restaurar
if _original_tracking_uri:
    os.environ["MLFLOW_TRACKING_URI"] = _original_tracking_uri

print(f"Entrenamiento completado: {results.save_dir}")

Ultralytics 8.4.41 🚀 Python-3.12.3 torch-2.7.1+cu126 CUDA:0 (NVIDIA A10G, 22588MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=toys-detection-v2, nbs=64, nms=False, opset=None, optimize=

/local_disk0/.ephemeral_nfs/envs/pythonEnv-ccb88140-4da9-480f-ab8f-1a492a20f530/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/23 01:16:59 INFO mlflow.tracking.fluent: Experiment with name '/tmp/yolo_training_v2' does not exist. Creating a new experiment.
2026/04/23 01:16:59 WARNING mlflow.tracking.fluent: Exception raised while enabling autologging for pyspark: MLflow Spark dataset autologging is not supported on Databricks shared clusters or Databricks serverless clusters.
2026/04/23 01:16:59 WARNING mlflow.tracking.fluent: Exception raised while enabling autologging for pyspark.ml: [JVM_ATT

MLflow: logging run_id(16855c2122b44dfe832370311cc1978a) to runs/mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs/mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /tmp/yolo_training_v2/toys-detection-v2
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50      2.43G     0.4953      6.061   0.007747         24        640: 100% ━━━━━━━━━━━━ 10/10 2.0s/it 19.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.2s/it 1.2s
                   all         15         16    0.00361          1      0.239      0.206

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50      2.45G     0.3268      5.255   0.004989         20        640: 100% ━━━━━━━━━━━━ 10/10 6.9it/s 1.5s
                 Class     Ima

2026/04/23 01:18:40 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/04/23 01:18:41 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


MLflow: results logged to runs/mlflow
MLflow: disable with 'yolo settings mlflow=False'
Entrenamiento completado: /tmp/yolo_training_v2/toys-detection-v2


In [0]:
mlflow.end_run()
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment(EXPERIMENT_NAME)

# Recoger estadisticas del Feature Store para tags
img_stats = df_image_features.agg(
    F.count("*").alias("total_images"),
    F.sum("num_objects").alias("total_annotations"),
).collect()[0]

with mlflow.start_run(run_name="v2_medallion_pipeline") as run:

    # 1. Parametros
    mlflow.log_params({
        "model": YOLO_MODEL,
        "epochs": EPOCHS,
        "patience": PATIENCE,
        "img_size": IMG_SIZE,
        "batch_size": BATCH_SIZE,
        "silver_table": SILVER_TABLE,
        "gold_image_table": GOLD_IMAGE_TABLE,
        "gold_annotation_table": GOLD_ANNOTATION_TABLE,
        "data_quality_ratio": f"{quality_ratio:.4f}",
        "pipeline_version": "v2",
    })

    # 2. Metricas por epoca
    results_csv = os.path.join(str(results.save_dir), "results.csv")
    df_results = pd.read_csv(results_csv)
    df_results.columns = df_results.columns.str.strip()

    for _, row in df_results.iterrows():
        epoch = int(row["epoch"])
        for col in df_results.columns:
            if col == "epoch":
                continue
            safe = col.replace("(", "").replace(")", "").replace("/", "_").strip()
            mlflow.log_metric(safe, float(row[col]), step=epoch)

    # 3. Metricas finales
    metrics = results.results_dict
    for k, v in metrics.items():
        safe = "final_" + k.replace("(", "").replace(")", "").replace("/", "_")
        mlflow.log_metric(safe, v)

    # 4. Tags con metadata del Feature Store
    mlflow.set_tags({
        "dataset.total_images": img_stats["total_images"],
        "dataset.total_annotations": img_stats["total_annotations"],
        "dataset.quality_ratio": f"{quality_ratio:.4f}",
        "pipeline": "medallion_v2",
        "feature_store.image_table": GOLD_IMAGE_TABLE,
        "feature_store.annotation_table": GOLD_ANNOTATION_TABLE,
    })

    # 5. Artefactos
    train_dir = str(results.save_dir)
    if os.path.exists(train_dir):
        mlflow.log_artifacts(train_dir, artifact_path="training_results")

    best_model_path = os.path.join(train_dir, "weights", "best.pt")
    if os.path.exists(best_model_path):
        mlflow.log_artifact(best_model_path, artifact_path="model")

    run_id = run.info.run_id
    print(f"\nMLflow Run ID: {run_id}")
    print(f"Metricas finales:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

2026/04/23 01:18:42 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/04/23 01:19:14 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/04/23 01:19:14 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!



MLflow Run ID: eed0acef6ba946648ba3d3af592510ea
Metricas finales:
  metrics/precision(B): 0.9738
  metrics/recall(B): 0.9989
  metrics/mAP50(B): 0.9950
  metrics/mAP50-95(B): 0.9890
  fitness: 0.9890


In [0]:
import mlflow.pyfunc
from mlflow.models.signature import ModelSignature
from mlflow.types import ColSpec, Schema

# Wrapper minimo para registrar YOLO como modelo MLflow
class _YOLOPyfuncModel(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        from ultralytics import YOLO
        self.model = YOLO(context.artifacts["best_weights"])

    def predict(self, context, model_input):
        return self.model(model_input)

# Loguear modelo correctamente dentro del run existente
with mlflow.start_run(run_id=run_id):
    signature = ModelSignature(
        inputs=Schema([ColSpec("string", "image_path")]),
        outputs=Schema([ColSpec("string", "detections")]),
    )
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=_YOLOPyfuncModel(),
        artifacts={"best_weights": best_model_path},
        pip_requirements=["ultralytics", "numpy==1.26.4"],
        signature=signature,
    )

# Registrar la run como un modelo versionado en UC
result = mlflow.register_model(
    model_uri=f"runs:/{run_id}/model",
    name=MODEL_NAME,
)

print(f"\nModelo registrado en Unity Catalog: {MODEL_NAME}")
print(f"  Version: {result.version}")
print(f"  Run ID:  {run_id}")

# Agregar alias al modelo
client = mlflow.tracking.MlflowClient()
client.set_registered_model_alias(MODEL_NAME, "champion", result.version)
print(f"  Alias:   champion -> v{result.version}")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-ccb88140-4da9-480f-ab8f-1a492a20f530/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/04/23 01:25:56 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/04/23 01:25:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://fevm-jgworkspaceclassic.cloud.databricks.com/ml/experiments/2564958131611974/models/m-30db97c53f534c00922eb35e77148b53?o=7474648405668119
/local_disk0/.ephemeral_nfs/envs/pythonEnv-ccb88140-4da9-480f-ab8f-1a492a20f530/lib/python3.12/site-packages/mlflow/pyfunc/__init__.py:3340: UserWarning: An input example was not provided when logging th

2026/04/23 01:25:59 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/04/23 01:25:59 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
Registered model 'jgworkspaceclassic_catalog.infra_demo.cv_v2_yolo_toys' already exists. Creating a new version of this model...
2026/04/23 01:26:00 WARNING mlflow.tracking._model_registry.fluent: Run with id eed0acef6ba946648ba3d3af592510ea has no artifacts at artifact path 'model', registering model based on models:/m-30db97c53f534c00922eb35e77148b53 instead


Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

🔗 Created version '1' of model 'jgworkspaceclassic_catalog.infra_demo.cv_v2_yolo_toys': https://fevm-jgworkspaceclassic.cloud.databricks.com/explore/data/models/jgworkspaceclassic_catalog/infra_demo/cv_v2_yolo_toys/version/1?o=7474648405668119



Modelo registrado en Unity Catalog: jgworkspaceclassic_catalog.infra_demo.cv_v2_yolo_toys
  Version: 1
  Run ID:  eed0acef6ba946648ba3d3af592510ea
  Alias:   champion -> v1


In [0]:
import shutil

best_weights_src = os.path.join(str(results.save_dir), "weights", "best.pt")
best_weights_dst = os.path.join(VOLUME_PATH, "models", "best_v2.pt")

os.makedirs(os.path.dirname(best_weights_dst), exist_ok=True)
shutil.copy2(best_weights_src, best_weights_dst)
print(f"Pesos copiados a: {best_weights_dst}")

Pesos copiados a: /Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/models/best_v2.pt
